# Simulación de Parqueadero - Supercentro
## Actividad Didáctica 2 - M4 | Modelo M/M/1

**Modelo:** M/M/1 — Colas independientes por cajero  
**Objetivo:** Determinar si 3 cajeros son suficientes para atender la demanda

## 1. Librerías e importaciones

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random

np.random.seed(42)
random.seed(42)

print('✓ Librerías importadas correctamente')

✓ Librerías importadas correctamente


## 2. Parámetros del modelo

In [2]:
# Tipos de usuario con sus parámetros (tiempos en minutos)
TIPOS_USUARIO = {
    'Rápido':    {'media_servicio': 1, 'media_llegada': 3,  'prob': 0.25},
    'Normal':    {'media_servicio': 3, 'media_llegada': 3,  'prob': 0.20},
    'Lento':     {'media_servicio': 4, 'media_llegada': 5,  'prob': 0.275},
    'Muy Lento': {'media_servicio': 6, 'media_llegada': 7,  'prob': 0.275},
}

NUM_CAJEROS   = 3
NUM_REPLICAS  = 300
CLIENTES_REP  = 500

# Mostrar factores de utilización ρ = λ/μ
print('Factor de utilización ρ por tipo de usuario:')
print('-' * 50)
for tipo, p in TIPOS_USUARIO.items():
    lam = 1/p['media_llegada']
    mu  = 1/p['media_servicio']
    rho = lam/mu
    est = '✓ Estable' if rho < 1 else '✗ INESTABLE (cola infinita)'
    print(f"  {tipo:12s}: ρ = {rho:.4f}  →  {est}")

Factor de utilización ρ por tipo de usuario:
--------------------------------------------------
  Rápido      : ρ = 0.3333  →  ✓ Estable
  Normal      : ρ = 1.0000  →  ✗ INESTABLE (cola infinita)
  Lento       : ρ = 0.8000  →  ✓ Estable
  Muy Lento   : ρ = 0.8571  →  ✓ Estable


## 3. Función de simulación M/M/1

In [3]:
def simular_mm1(media_llegada, media_servicio, n_clientes):
    """
    Simula una cola M/M/1.
    Retorna: (tiempo_medio_espera_Wq, tiempo_medio_sistema_W)
    """
    tiempos_llegada  = np.cumsum(np.random.exponential(media_llegada, n_clientes))
    tiempos_servicio = np.random.exponential(media_servicio, n_clientes)
    
    esperas = []
    sistemas = []
    fin_servicio = 0.0
    
    for i in range(n_clientes):
        inicio    = max(tiempos_llegada[i], fin_servicio)
        espera    = inicio - tiempos_llegada[i]
        fin_servicio = inicio + tiempos_servicio[i]
        esperas.append(espera)
        sistemas.append(espera + tiempos_servicio[i])
    
    return np.mean(esperas), np.mean(sistemas)

print('✓ Función de simulación M/M/1 definida')

✓ Función de simulación M/M/1 definida


## 4. Ejecución de réplicas

In [4]:
resultados = {}

for tipo, params in TIPOS_USUARIO.items():
    esperas, sistemas = [], []
    for _ in range(NUM_REPLICAS):
        e, s = simular_mm1(params['media_llegada'], params['media_servicio'], CLIENTES_REP)
        esperas.append(e)
        sistemas.append(s)
    resultados[tipo] = {
        'esperas':  np.array(esperas),
        'sistemas': np.array(sistemas)
    }
    print(f'✓ {tipo}: {NUM_REPLICAS} réplicas completadas')

print('\n✓ Simulación completa')

✓ Rápido: 300 réplicas completadas
✓ Normal: 300 réplicas completadas
✓ Lento: 300 réplicas completadas
✓ Muy Lento: 300 réplicas completadas

✓ Simulación completa


## 5. Determinación del Estado Estable
Se usa la técnica de **media acumulada** con umbral de coeficiente de variación < 5%

In [ ]:
def determinar_estado_estable(medias, ventana=20, umbral=0.05):
    """
    Identifica la réplica desde la cual el sistema alcanza estado estable.
    Criterio: coeficiente de variación de la media acumulada < umbral.
    """
    promedios_acum = np.cumsum(medias) / (np.arange(len(medias)) + 1)
    for i in range(ventana, len(promedios_acum)):
        segmento = promedios_acum[i-ventana:i]
        cv = np.std(segmento) / (np.mean(segmento) + 1e-9)
        if cv < umbral:
            return i - ventana
    return ventana

estados_estables = {}
for tipo, datos in resultados.items():
    ee = determinar_estado_estable(datos['sistemas'])
    estados_estables[tipo] = ee
    print(f'{tipo:12s} → Estado estable desde réplica: {ee}')

# ── Gráfica de estado estable ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Determinación del Estado Estable\nTiempo Promedio en Sistema (W)', fontsize=14, fontweight='bold')
colores = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

for ax, (tipo, datos), color in zip(axes.flatten(), resultados.items(), colores):
    medias       = datos['sistemas']
    prom_acum    = np.cumsum(medias) / (np.arange(len(medias)) + 1)
    ee           = estados_estables[tipo]
    
    ax.plot(prom_acum, color=color, linewidth=2, label='Media acumulada')
    ax.axvline(x=ee, color='red', linestyle='--', linewidth=1.8, label=f'Estado estable: réplica {ee}')
    ax.fill_between(range(len(prom_acum)), prom_acum, alpha=0.12, color=color)
    ax.set_title(f'Usuario {tipo}', fontweight='bold')
    ax.set_xlabel('Número de réplicas')
    ax.set_ylabel('Tiempo promedio en sistema (min)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('estado_estable.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Gráfica guardada: estado_estable.png')

Rápido       → Estado estable desde réplica: 0
Normal       → Estado estable desde réplica: 9
Lento        → Estado estable desde réplica: 2
Muy Lento    → Estado estable desde réplica: 14


## 6. Estadísticas post estado estable

In [ ]:
stats_finales = {}

for tipo, datos in resultados.items():
    ee          = estados_estables[tipo]
    esperas_ee  = datos['esperas'][ee:]
    sistemas_ee = datos['sistemas'][ee:]
    
    stats_finales[tipo] = {
        'ee':              ee,
        'media_espera':    np.mean(esperas_ee),
        'media_sistema':   np.mean(sistemas_ee),
        'std_espera':      np.std(esperas_ee),
        'std_sistema':     np.std(sistemas_ee),
        'replicas_validas': len(esperas_ee),
        'esperas_ee':      esperas_ee,
        'sistemas_ee':     sistemas_ee,
    }

# Tabla de resultados
df_stats = pd.DataFrame({
    'Tipo':            list(stats_finales.keys()),
    'Estado estable':  [s['ee'] for s in stats_finales.values()],
    'Réplicas válidas':[s['replicas_validas'] for s in stats_finales.values()],
    'Wq (min)':        [round(s['media_espera'],4) for s in stats_finales.values()],
    'W (min)':         [round(s['media_sistema'],4) for s in stats_finales.values()],
    'σ W':             [round(s['std_sistema'],4) for s in stats_finales.values()],
})
print(df_stats.to_string(index=False))

min_t = min(stats_finales, key=lambda t: stats_finales[t]['media_sistema'])
max_t = max(stats_finales, key=lambda t: stats_finales[t]['media_sistema'])
print(f"\n→ Tipo con MENOR tiempo promedio: {min_t} (W = {stats_finales[min_t]['media_sistema']:.4f} min)")
print(f"→ Tipo con MAYOR tiempo promedio: {max_t} (W = {stats_finales[max_t]['media_sistema']:.4f} min)")

## 7. Promedio de usuarios por cajero

In [ ]:
np.random.seed(100)
resultados_cajeros = []

for cajero in range(1, NUM_CAJEROS + 1):
    fila = {'Cajero': cajero}
    for tipo, params in TIPOS_USUARIO.items():
        ee = estados_estables[tipo]
        esperas, sistemas = [], []
        for _ in range(NUM_REPLICAS - ee):
            e, s = simular_mm1(params['media_llegada'], params['media_servicio'], CLIENTES_REP)
            esperas.append(e)
            sistemas.append(s)
        fila[f'Wq_{tipo}'] = np.mean(esperas)
        fila[f'W_{tipo}']  = np.mean(sistemas)
    resultados_cajeros.append(fila)

df_cajeros = pd.DataFrame(resultados_cajeros)

print('Tiempo promedio en sistema W (min) por cajero y tipo de usuario:')
print('-' * 65)
for _, row in df_cajeros.iterrows():
    print(f"Cajero {int(row['Cajero'])}: ", end='')
    for tipo in TIPOS_USUARIO.keys():
        print(f"{tipo}={row[f'W_{tipo}']:.2f}  ", end='')
    print()

# Gráfica de barras
fig, ax = plt.subplots(figsize=(12, 6))
x     = np.arange(NUM_CAJEROS)
width = 0.2
tipos_keys = list(TIPOS_USUARIO.keys())

for i, (tipo, color) in enumerate(zip(tipos_keys, colores)):
    vals = df_cajeros[f'W_{tipo}']
    ax.bar(x + i*width, vals, width, label=tipo, color=color, alpha=0.85)

ax.set_xlabel('Cajero', fontsize=12)
ax.set_ylabel('Tiempo promedio en sistema W (min)', fontsize=12)
ax.set_title('Tiempo Promedio de Atención por Cajero y Tipo de Usuario', fontsize=13, fontweight='bold')
ax.set_xticks(x + width*1.5)
ax.set_xticklabels([f'Cajero {i+1}' for i in range(NUM_CAJEROS)])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('comparacion_cajeros.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Verificación y Calibración del modelo
Comparación entre valores teóricos M/M/1 y simulados

In [ ]:
print('Verificación: Valores teóricos vs simulados')
print('=' * 65)

W_teos  = []
W_sims  = []
rhos    = []

for tipo, params in TIPOS_USUARIO.items():
    lam = 1/params['media_llegada']
    mu  = 1/params['media_servicio']
    rho = lam/mu
    rhos.append(rho)
    if rho < 1:
        Wq    = rho / (mu*(1-rho))
        W_teo = Wq + params['media_servicio']
        err   = abs(W_teo - stats_finales[tipo]['media_sistema'])
        pct   = err/W_teo*100
        print(f"{tipo:12s}: ρ={rho:.3f}  W_teo={W_teo:.3f}  W_sim={stats_finales[tipo]['media_sistema']:.3f}  Error={pct:.2f}%")
    else:
        W_teo = float('inf')
        print(f"{tipo:12s}: ρ={rho:.3f}  W_teo=∞ (cola inestable)  W_sim={stats_finales[tipo]['media_sistema']:.3f}")
    W_teos.append(min(W_teo, 60))
    W_sims.append(stats_finales[tipo]['media_sistema'])

# Gráficas de calibración
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Verificación y Calibración del Modelo M/M/1', fontsize=13, fontweight='bold')

# Teórico vs simulado
x   = np.arange(len(TIPOS_USUARIO))
w05 = 0.35
axes[0].bar(x - w05/2, W_teos, w05, label='Teórico M/M/1', color='#1976D2', alpha=0.85)
axes[0].bar(x + w05/2, W_sims, w05, label='Simulado',      color='#43A047', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(list(TIPOS_USUARIO.keys()))
axes[0].set_title('Teórico vs Simulado'); axes[0].legend()
axes[0].set_ylabel('W (min)'); axes[0].grid(True, alpha=0.3, axis='y')

# Factor de utilización
bcolors = ['#43A047' if r < 1 else '#E53935' for r in rhos]
bars = axes[1].bar(list(TIPOS_USUARIO.keys()), rhos, color=bcolors, alpha=0.85)
axes[1].axhline(y=1, color='red', linestyle='--', linewidth=2, label='ρ = 1 (límite estabilidad)')
axes[1].set_title('Factor de Utilización ρ = λ/μ')
axes[1].set_ylabel('ρ'); axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')
for bar, r in zip(bars, rhos):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{r:.3f}', ha='center')

plt.tight_layout()
plt.savefig('calibracion.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Eliminación del estado transitorio — Antes y Después

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Eliminación del Estado Transitorio\nTiempo Promedio en Sistema W (min)', 
             fontsize=13, fontweight='bold')

titulos  = ['ANTES (incluye estado transitorio)', 'DESPUÉS (sin estado transitorio)']
usar_ees = [False, True]

for ax, titulo, usar_ee in zip(axes, titulos, usar_ees):
    for (tipo, datos), color in zip(resultados.items(), colores):
        serie = datos['sistemas']
        if usar_ee:
            serie = serie[estados_estables[tipo]:]
        prom_acum = np.cumsum(serie) / (np.arange(len(serie)) + 1)
        ax.plot(prom_acum, color=color, linewidth=2, label=tipo, alpha=0.9)
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('Número de réplicas')
    ax.set_ylabel('Tiempo promedio acumulado (min)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('antes_despues.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla comparativa
print('\nComparación antes/después de eliminar estado transitorio:')
print('-'*60)
for tipo, datos in resultados.items():
    W_con    = np.mean(datos['sistemas'])
    W_sin    = stats_finales[tipo]['media_sistema']
    diferencia = W_con - W_sin
    print(f"{tipo:12s}: Con transitorio={W_con:.3f}  Sin={W_sin:.3f}  Δ={diferencia:+.3f} min")

## 10. Conclusiones y Estrategia

In [ ]:
print('=' * 65)
print('CONCLUSIONES Y ESTRATEGIA')
print('=' * 65)

for tipo, params in TIPOS_USUARIO.items():
    lam = 1/params['media_llegada']
    mu  = 1/params['media_servicio']
    rho = lam/mu
    W   = stats_finales[tipo]['media_sistema']
    if rho < 0.6:
        rec = '✅ 3 cajeros SUFICIENTES'
    elif rho < 1:
        rec = '⚠️  3 cajeros ACEPTABLES, monitorear en horas pico'
    else:
        rec = '❌ SE REQUIEREN MÁS CAJEROS'
    print(f"  {tipo:12s}: ρ={rho:.3f}  W={W:.2f}min  →  {rec}")

print()
print('ESTRATEGIA RECOMENDADA:')
print('  1. Agregar 1 cajero adicional para usuarios Normales (ρ ≥ 1)')
print('  2. Implementar pagos digitales/app para reducir tiempo de servicio')
print('  3. Capacitar operadores para agilizar interacción usuario-cajero')
print('  4. Monitorear usuarios Muy Lentos en horas pico (ρ = 0.857)')